# NBA Coach's Challenge & Replay Archive Classifier (`ChallengeReplayClassifier`)
**Source:** [official.nba.com/replay/archive](https://official.nba.com/replay/archive/)  
**Task:** Context-Conditioned Multi-Head Challenge Replay Classification from multi-angle 60 fps NBA Replay Center footage.

### Pipeline Overview
1. **Stage 1 (`1_scrape_replays.py`)**: Queries `/wp-json/api/v1/query_replay_feed` across seasons for all Coach's Challenge types (`challenge_foul`, `challenge_oob`, `challenge_goaltending_bi`), parses verbatim Replay Center rulings into the 8 specific call outcomes (`foul_stands`, `foul_overturned_no_foul`, `foul_overturned_other_foul`, `oob_stands`, `oob_overturned`, `oob_proximate_foul`, `goaltending_bi_stands`, `goaltending_bi_overturned`), and downloads MP4 clips.
2. **Stage 2 (`2_preprocess_replays.py`)**: Segments multi-angle 60 fps replay clips by hard camera cuts, filters out low-saliency dead-ball shots, and extracts up to `K` shot-bounded `(3, 16, 224, 224)` RGB + `(16, 16)` biomechanical & court-context action tubes per play.
3. **Stage 3 (`3_model_and_dataset.py`)**: Dual-stream `ChallengeReplayClassifier` (`R(2+1)D-18` visual + `PoseTemporalEncoder` 16-D attention + `challenge_type_head` + `challenge_context_embed` + 8-class `call_head` + `resolution_head` + `foul_head`) with leak-free `ReplayFoulDataset`.
4. **Stage 4 (`4_train_eval.py`)**: Game-grouped Train/Val split (`GroupShuffleSplit`), saliency-weighted multi-angle play evaluation, threshold calibration, and held-out `data/test/` inference.


In [ ]:
# ==============================================================================
# CELL 1: Environment Bootstrap (Google Colab / Local) & Pipeline Config
# ==============================================================================
import csv
import importlib
import json
import logging
import os
import subprocess
import sys
from collections import Counter
from pathlib import Path

# Silence noisy urllib3 retry warnings when official.nba.com rate-limits HTML requests
logging.getLogger("urllib3").setLevel(logging.ERROR)
logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    print("Running in Google Colab — installing dependencies & syncing `testing` branch...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "opencv-python-headless", "scikit-learn", "matplotlib", "beautifulsoup4", "requests"],
        check=False,
    )
    repo_dir = Path("/content/AI-ML-Final-Project")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "-b", "testing", "https://github.com/ferraromike/AI-ML-Final-Project.git", str(repo_dir)],
            check=False,
        )
    else:
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", "testing"], check=False)
        subprocess.run(["git", "-C", str(repo_dir), "reset", "--hard", "origin/testing"], check=False)
    target_dir = repo_dir / "NBA-Replay-Foul-Classifier"
    os.chdir(str(target_dir if target_dir.exists() else repo_dir))

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("YOLO_CONFIG_DIR", "/tmp/Ultralytics")

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve

CWD = Path.cwd().resolve()
if (CWD / "NBA-Replay-Foul-Classifier").exists():
    PROJECT_DIR = CWD / "NBA-Replay-Foul-Classifier"
elif (CWD / "1_scrape_replays.py").exists():
    PROJECT_DIR = CWD
elif (CWD.parent / "NBA-Replay-Foul-Classifier").exists():
    PROJECT_DIR = CWD.parent / "NBA-Replay-Foul-Classifier"
else:
    PROJECT_DIR = CWD

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Reload pipeline modules if already imported in an active kernel session
for _mod_name in ("1_scrape_replays", "2_preprocess_replays", "3_model_and_dataset", "4_train_eval"):
    if _mod_name in sys.modules:
        importlib.reload(sys.modules[_mod_name])

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_CSV = DATA_DIR / "replay_metadata.csv"
PROCESSED_METADATA_CSV = PROCESSED_DIR / "processed_metadata.csv"
TEST_DIR = DATA_DIR / "test"
ANNOTATED_DIR = DATA_DIR / "annotated"
WEIGHTS_DIR = PROJECT_DIR / "weights"
CHECKPOINT_PATH = WEIGHTS_DIR / "best_replay_foul_classifier.pt"

IS_CUDA = torch.cuda.is_available()
VRAM_GB = (torch.cuda.get_device_properties(0).total_memory / 1e9) if IS_CUDA else 0.0
DEVICE = torch.device("cuda" if IS_CUDA else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"))

CONFIG = {
    # Stage 1: Full Replay Archive Scraper (foul, oob, goaltending, flagrant)
    "seasons": ["2026-27", "2025-26", "2024-25"],
    "include_flagrant_reviews": True,       # Scrape Flagrant reviews into data/raw/flagrant/
    "include_non_foul_challenges": True,     # Scrape Foul, Out of Bounds, and Goaltending/BI challenges
    "fast_cdn_only": True,                   # True = skip HTML scraping if replay_metadata.csv already has >= max_clips rows
    "max_inspect": 400 if IS_CUDA else 160,  # Uses 10x faster bulk RSS search feeds when scraping new pages
    "max_clips": 600 if IS_CUDA else 120,    # Increase to 1000+ or 0 (all 1,373+ pre-indexed plays) on Colab
    "quality": "hd" if IS_CUDA else "sd",    # Tip: use 'sd' on Colab to download & preprocess 1000+ clips 4x faster
    "scrape_workers": 12,
    # Stage 2: Shot-Segmented Multi-Angle 16-D Preprocessing
    "angles_per_clip": 2,
    "window_sec": 1.6,
    "num_frames": 16,
    "crop_size": 224,
    "scan_fps": 5.0,
    "max_preprocess_clips": 0,               # 0 = process all downloaded raw MP4s
    "yolo_batch_size": 64 if (IS_CUDA and VRAM_GB >= 12.0) else 16,
    # Stage 3 & 4: Multi-Head Challenge Replay Training
    "backbone_mode": "partial" if (IS_CUDA and VRAM_GB >= 10.0) else "frozen",
    "epochs": 20 if IS_CUDA else 18,
    "batch_size": 8 if (IS_CUDA and VRAM_GB >= 12.0) else 4,
    "accum_steps": 2,
    "lr": 3e-4,
    "weight_decay": 1e-2,
    "dropout": 0.30,
    "modality_dropout": 0.08,
    "val_split": 0.20,
    "seed": 42,
}

print(f"Project Dir  : {PROJECT_DIR}")
print(f"Device       : {DEVICE} (VRAM={VRAM_GB:.1f} GB)")
print(f"Backbone Mode: {CONFIG['backbone_mode']} | Quality: {CONFIG['quality']} | Max Clips: {CONFIG['max_clips']} (0=all)")


---
## Stage 1: Scrape Official NBA Replay Archive (`1_scrape_replays.py`)
Scrapes Coach's Challenges (`Challenge of Called Foul`, `Challenge OOB`, `Challenge Goaltending/BI`), parses each permalink's verbatim Replay Center ruling into both `challenge_type` and the 8 specific call outcomes (`call_label`), and downloads MP4 clips.


In [ ]:
# ==============================================================================
# CELL 2: Stage 1 — Scrape official.nba.com/replay/archive/ & Download Clips
# ==============================================================================
scrape_mod = importlib.import_module("1_scrape_replays")
importlib.reload(scrape_mod)

existing_rows = scrape_mod.load_existing_metadata(METADATA_CSV)
target_clips = CONFIG["max_clips"]
if CONFIG.get("fast_cdn_only", True) and existing_rows and (target_clips > 0 and len(existing_rows) >= target_clips):
    print(f"Fast CDN Mode: Using {len(existing_rows)} pre-indexed plays from {METADATA_CSV.name} (0s HTML scraping)...")
    scrape_mod.bulk_download_clips(
        rows=existing_rows,
        output_dir=RAW_DIR,
        max_clips=target_clips,
        max_workers=CONFIG["scrape_workers"],
        balanced=True,
    )
    rows = existing_rows
else:
    rows = scrape_mod.run_replay_scraper(
        output_dir=RAW_DIR,
        metadata_csv=METADATA_CSV,
        seasons=CONFIG["seasons"],
        include_flagrant_reviews=CONFIG["include_flagrant_reviews"],
        include_non_foul_challenges=CONFIG["include_non_foul_challenges"],
        max_inspect=CONFIG["max_inspect"],
        max_clips=CONFIG["max_clips"],
        quality=CONFIG["quality"],
        workers=CONFIG["scrape_workers"],
        skip_download=False,
    )

folder_counts = {
    c: len(list((RAW_DIR / c).glob("*.mp4"))) if (RAW_DIR / c).exists() else 0
    for c in ("foul", "oob", "goaltending", "flagrant")
}
print(f"Metadata rows : {len(rows)}")
print(f"Raw MP4 clips by challenge type: {folder_counts} ({sum(folder_counts.values())} total)")


---
## Exploratory Data Analysis (EDA): Challenge Types & Specific Call Outcomes


In [ ]:
# ==============================================================================
# CELL 3: EDA — Challenge Types, 8 Specific Call Outcomes & Binary Foul Balance
# ==============================================================================
scrape_mod = importlib.import_module("1_scrape_replays")
importlib.reload(scrape_mod)

def _normalize_row(row):
    if hasattr(scrape_mod, "upgrade_metadata_row"):
        return scrape_mod.upgrade_metadata_row(dict(row))
    r = dict(row)
    title_low = str(r.get("title", "")).lower()
    if not r.get("challenge_type"):
        if "oob" in title_low or "out of bounds" in title_low:
            r["challenge_type"] = "challenge_oob"
        elif "goaltending" in title_low or "interference" in title_low:
            r["challenge_type"] = "challenge_goaltending_bi"
        else:
            r["challenge_type"] = "challenge_foul"
    return r

with open(METADATA_CSV, "r", encoding="utf-8") as f:
    meta_rows = [_normalize_row(r) for r in csv.DictReader(f)]

lbl_counts = Counter(int(r["label"]) for r in meta_rows)
ctype_counts = Counter(r.get("challenge_type", "challenge_foul") for r in meta_rows)
out_counts = Counter(r.get("outcome_category", "unknown") for r in meta_rows)

print(f"Total Replay Archive entries: {len(meta_rows)}")
print(f"  Challenge Types : {dict(ctype_counts)}")
print(f"  Call Outcomes   : {dict(out_counts)}")
print(f"  FOUL (1)        : {lbl_counts[1]} ({100.0 * lbl_counts[1] / max(len(meta_rows), 1):.1f}%)")
print(f"  NO FOUL (0)     : {lbl_counts[0]} ({100.0 * lbl_counts[0] / max(len(meta_rows), 1):.1f}%)")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), dpi=100)
ct_keys = list(ctype_counts.keys())
axes[0].bar(ct_keys, [ctype_counts[k] for k in ct_keys], color="#9467bd")
axes[0].set_title("What Is Being Challenged", fontweight="bold")
axes[0].tick_params(axis="x", rotation=15)

out_keys = list(out_counts.keys())
axes[1].barh(out_keys, [out_counts[k] for k in out_keys], color="#1f77b4")
axes[1].set_title("Parsed Replay Center Rulings", fontweight="bold")

axes[2].bar(["No Foul (0)", "Foul (1)"], [lbl_counts[0], lbl_counts[1]], color=["#2ca02c", "#d62728"], width=0.5)
axes[2].set_title("Binary Foul Indicator", fontweight="bold")

plt.tight_layout()
plt.show()


---
## Stage 2: Shot-Segmented Multi-Angle 16-D Court & Pose Preprocessing (`2_preprocess_replays.py`)


In [ ]:
# ==============================================================================
# CELL 4: Stage 2 — Extract Multi-Angle Shot-Bounded Action Tubes & 16-D Features
# ==============================================================================
preprocess_mod = importlib.import_module("2_preprocess_replays")
importlib.reload(preprocess_mod)

processed_rows = preprocess_mod.preprocess_replay_dataset(
    metadata_csv=METADATA_CSV,
    processed_dir=PROCESSED_DIR,
    processed_metadata_csv=PROCESSED_METADATA_CSV,
    test_dir=TEST_DIR,
    angles_per_clip=CONFIG["angles_per_clip"],
    base_window_sec=CONFIG["window_sec"],
    num_frames=CONFIG["num_frames"],
    crop_size=CONFIG["crop_size"],
    scan_fps=CONFIG["scan_fps"],
    max_clips=CONFIG["max_preprocess_clips"],
    yolo_batch_size=CONFIG["yolo_batch_size"],
    device="cuda" if IS_CUDA else "cpu",
    synthetic_fallback=False,
)

unique_plays = {r["play_id"] for r in processed_rows}
unique_games = {r["game_id"] for r in processed_rows}
print(f"Processed tubes: {len(processed_rows)} across {len(unique_plays)} plays & {len(unique_games)} games")


---
## Stage 3: Context-Conditioned Multi-Head Classifier Verification (`3_model_and_dataset.py`)


In [ ]:
# ==============================================================================
# CELL 5: Stage 3 — Inspect Classifier & Verify Zero-Leakage Split
# ==============================================================================
model_mod = importlib.import_module("3_model_and_dataset")
train_mod = importlib.import_module("4_train_eval")
importlib.reload(model_mod)
importlib.reload(train_mod)

use_cached = CONFIG["backbone_mode"] == "frozen"
base_ds = model_mod.ReplayFoulDataset(
    processed_dir=PROCESSED_DIR,
    metadata_csv=PROCESSED_METADATA_CSV,
    test_dir=TEST_DIR,
    is_train=True,
    use_cached_embeddings=use_cached,
)
train_recs, val_recs = train_mod.split_records_grouped(
    base_ds.records, val_split=CONFIG["val_split"], seed=CONFIG["seed"]
)

tr_plays = {r["play_id"] for r in train_recs}
va_plays = {r["play_id"] for r in val_recs}
print(f"Dataset tubes : {len(base_ds)} | Train: {len(train_recs)} ({len(tr_plays)} plays) | Val: {len(val_recs)} ({len(va_plays)} plays)")
print(f"Play overlap  : {len(tr_plays & va_plays)} (guaranteed 0)")

ModelCls = getattr(model_mod, "ChallengeReplayClassifier", model_mod.BinaryFoulClassifier)
probe_model = ModelCls(
    pretrained=True,
    backbone_mode=CONFIG["backbone_mode"],
    dropout_p=CONFIG["dropout"],
    modality_dropout=CONFIG["modality_dropout"],
)
summary = probe_model.trainable_parameter_summary()
print(f"Model Params  : Total={summary['total_params']:,} | Trainable={summary['trainable_params']:,} | Frozen={summary['frozen_params']:,}")
del probe_model


---
## Stage 4: Train Multi-Head Challenge Replay Classifier (`4_train_eval.py`)


In [ ]:
# ==============================================================================
# CELL 6: Stage 4 — Train Multi-Head ChallengeReplayClassifier & Plot Metrics
# ==============================================================================
from argparse import Namespace

train_args = Namespace(
    processed_dir=PROCESSED_DIR,
    metadata_csv=PROCESSED_METADATA_CSV,
    test_dir=TEST_DIR,
    annotated_dir=ANNOTATED_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    accum_steps=CONFIG["accum_steps"],
    max_samples=0,
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    dropout=CONFIG["dropout"],
    modality_dropout=CONFIG["modality_dropout"],
    val_split=CONFIG["val_split"],
    num_workers=2 if IS_CUDA else 0,
    device="auto",
    backbone_mode=CONFIG["backbone_mode"],
    no_cache=False,
    no_pretrained=False,
    seed=CONFIG["seed"],
)

train_out = train_mod.run_training_pipeline(train_args)
history = train_out["history"]
best_m = train_out["best_metrics"]
print("\nBest Play-Level Validation Metrics:", json.dumps(best_m, indent=2))

ep_x = [h["epoch"] for h in history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.2), dpi=100)
ax1.plot(ep_x, [h["train_loss"] for h in history], label="Train Loss", marker="o")
ax1.plot(ep_x, [h["val_loss"] for h in history], label="Val Loss", marker="s")
ax1.set_title("Multi-Task Challenge Replay Loss", fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(ep_x, [h["f1"] for h in history], label="Foul F1", marker="o")
ax2.plot(ep_x, [h.get("call_accuracy", h["balanced_accuracy"]) for h in history], label="Specific Call / Bal Acc", marker="s")
ax2.plot(ep_x, [h["roc_auc"] for h in history], label="ROC-AUC", marker="^")
ax2.set_title("Play-Level Validation Metrics", fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylim(0.0, 1.05)
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()


---
## Validation Deep Dive: Confusion Matrix, ROC Curve & Specific Call Breakdown


In [ ]:
# ==============================================================================
# CELL 7: Play-Level Confusion Matrix, ROC Curve & Challenge Call Breakdown
# ==============================================================================
ckpt = model_mod.load_torch_file_safe(CHECKPOINT_PATH, map_location="cpu")
val_preds = ckpt.get("val_play_predictions", [])
opt_thresh = float(ckpt.get("optimal_threshold", 0.50))

y_true = np.array([p["true_label"] for p in val_preds], dtype=int)
y_prob = np.array([p["foul_prob"] for p in val_preds], dtype=float)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), dpi=100)
cm = ckpt["metrics"]["confusion_matrix"]
cm_mat = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_mat, display_labels=["No Foul (0)", "Foul (1)"])
disp.plot(ax=ax1, cmap="Blues", colorbar=False)
ax1.set_title(f"Play-Level Confusion Matrix (thresh={opt_thresh:.2f})", fontweight="bold")

if len(np.unique(y_true)) > 1:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ax2.plot(fpr, tpr, lw=2, label=f"ROC-AUC = {ckpt['metrics']['roc_auc']:.3f}")
    ax2.plot([0, 1], [0, 1], "--", color="gray")
    ax2.set_xlabel("False Positive Rate")
    ax2.set_ylabel("True Positive Rate")
    ax2.set_title("Play-Level ROC Curve", fontweight="bold")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

plt.tight_layout()
plt.show()

for p in val_preds[:12]:
    ct_str = p.get("true_challenge_type", "challenge_foul")
    tc_str = p.get("true_call", "FOUL" if p["true_label"] == 1 else "NO FOUL")
    pc_str = p.get("pred_call", "FOUL" if p["foul_prob"] >= opt_thresh else "NO FOUL")
    print(f"  {p['play_id']} | Challenged: {ct_str} | True: {tc_str} vs Pred: {pc_str} | P(foul)={p['foul_prob']:.2f}")


---
## Held-Out Test Inference (`data/test/` -> `data/annotated/`)
Identifies what is being challenged on each test clip and determines the exact challenge ruling and correct call.


In [ ]:
# ==============================================================================
# CELL 8: Held-Out Test Inference & Interactive Challenge Verdict Console
# ==============================================================================
import base64
from IPython.display import HTML, display

TEST_DIR.mkdir(parents=True, exist_ok=True)
test_videos = train_mod.discover_video_files(TEST_DIR)
if not test_videos:
    print(f"No clips in {TEST_DIR}. Drop .mp4 replay clips into data/test/ and re-run this cell.")
else:
    reports = train_mod.predict_folder(
        folder_path=TEST_DIR,
        checkpoint_path=CHECKPOINT_PATH,
        annotated_dir=ANNOTATED_DIR,
        angles_per_clip=CONFIG["angles_per_clip"],
        render_video=True,
        clean_no_foul=False,
    )
    for rep in reports:
        color = "#d62728" if rep["is_foul"] else "#2ca02c"
        ct_disp = rep.get("challenge_type_display", "CHALLENGE OF CALLED FOUL")
        call_str = rep.get("correct_call", rep["verdict"])
        conf_val = rep.get("call_confidence", rep["foul_prob"]) * 100.0
        print(f"{rep['clip']} | Challenged: {ct_disp} -> {rep['verdict']} | Correct Call: {call_str} | P(foul)={rep['foul_prob']*100:.1f}%")
        mp4_p = Path(rep["output_mp4"]) if rep.get("output_mp4") else None
        if mp4_p and mp4_p.exists() and mp4_p.stat().st_size < 25_000_000:
            b64 = base64.b64encode(mp4_p.read_bytes()).decode("ascii")
            display(HTML(f'''
                <div style="border:2px solid {color}; border-radius:8px; padding:10px; margin:10px 0; max-width:760px;">
                    <div style="font-weight:bold; color:{color}; margin-bottom:4px;">
                        {rep['clip']} — {ct_disp}: {rep['verdict']} ({conf_val:.1f}%)
                    </div>
                    <div style="font-size:13px; margin-bottom:6px;">Correct Call: {call_str}</div>
                    <video width="720" controls>
                        <source src="data:video/mp4;base64,{b64}" type="video/mp4">
                    </video>
                </div>
            '''))
